# Event Selection (batched / live)

Interactive full-sample (or partial-sample) event selection using the same
`build_pipeline()` as CAF makers and systematics.

Processes `sel_all` `.df` files in rounds, refreshes a multi-panel of selection
overlays, and writes `merged_histdata.pkl` for summary / efficiency follow-up.

Thresholds and cut order: `makedf/selections.py` and `event_selection_pipeline_def.py`.

For cut development / threshold scans on a few files, use `event_selection.ipynb`.

Cluster (non-interactive): `scripts/run_event_selection_batched.py` or
`run_event_selection_batched.sh`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# print avaialbe memory
import psutil

vmem = psutil.virtual_memory()
print(f"Available memory: {vmem.available / 1024 ** 3:.2f} GiB / {vmem.total / 1024 ** 3:.2f} GiB total")
print(f"Used: {vmem.used / 1024 ** 3:.2f} GiB ({vmem.percent}%)")

In [ ]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.files_config import *
from analysis_village.numucc_1p0pi.makedf.selections import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)

## Configs


In [ ]:
# ===== files =====
base_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"

mc_dir = "2026_09_04_172912__sel_all-mc-CV_updated"
mc_dirt_dir = "2026_09_01_135047__sel_all-mc-dirt_updated"
mc_intime_dir = "2026_09_01_135508__sel_all-mc-Intime_updated"
data_dir = "2026_09_04_172930__sel_all-data-1e20_updated"
data_offbeam_dir = "2026_09_01_140024__sel_all-data-OffBeamLight_updated"

import glob
# print how many *df files are in each directory
for this_dir in [mc_dir, mc_dirt_dir, mc_intime_dir, data_dir, data_offbeam_dir]:
    print(f"{this_dir}: {len(glob.glob(f'{base_dir}/{this_dir}/*.df'))}")

In [ ]:
# ===== plot configs =====
from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
today_str = "GiBUU"
syst_tag = ""

# Optional overrides (None → dated work dir under /exp/sbnd/data/users/$USER/...)
plots_dir = path.join(PLOTS_BASE, f"event_selection-batched-{syst_tag}-{today_str}")


## Live batched knobs


In [ ]:
# --- live / partial-run knobs ---
MAX_FILES_PER_SAMPLE = None  # int, or None for all files
UPDATE_EVERY_N_FILES = 20  # refresh inline panel every N files (also at the end)
FILES_PER_JOB = None         # None → same as UPDATE_EVERY_N_FILES (concat load)
FILENAME_STR = "sel_all"     # same filter as legacy SampleBundle.load_from_dirs
SHOW_IN_NOTEBOOK = True      # live panel via IPython display (not a file)
SAVE_MERGED_PAYLOAD = True   # write merged_histdata.pkl (reloadable counts)
SAVE_FINAL_PANEL = False     # optional PNG under plots_dir
INCLUDE_EFFICIENCY = True    # append efficiency curves to the live panel
COSMIC_ESTIMATE = "intime"   # "intime" | "offbeam" — cosmic sample in live panel + follow-up plots

# Same layout as the files config cell / legacy load
BATCH_SAMPLE_DIRS = {
    "mc":      mc_dir,
    "data":    data_dir,
    "intime":  mc_intime_dir,
    "offbeam": data_offbeam_dir,
    "dirt":    mc_dirt_dir,
}

batch_work_base = None     # None → default under /exp/sbnd/data/users/$USER/...
batch_plots_dir = plots_dir


In [ ]:
from analysis_village.numucc_1p0pi.event_selection_live import (
    LiveAccumulateConfig,
    run_live_accumulate,
    replot_from_payload,
)
from analysis_village.numucc_1p0pi.event_selection_batched import (
    EventSelectionBatchedConfig,
    discover_jobs,
    run_full,
    show_saved_plots,
    default_event_selection_batched_work_root,
)

live_cfg = LiveAccumulateConfig(
    max_files_per_sample=MAX_FILES_PER_SAMPLE,
    files_per_job=FILES_PER_JOB,
    one_file_per_job=False,
    concat_load=True,
    update_every_round=True,
    update_every_n_files=UPDATE_EVERY_N_FILES,
    base_dir=base_dir,
    sample_dirs=BATCH_SAMPLE_DIRS,
    filename_str=FILENAME_STR,
    work_base=batch_work_base or default_event_selection_batched_work_root(
        f"live-{today_str}"
    ),
    plots_dir=batch_plots_dir,
    mc_univ_syst=("Flux", "G4", "GENIE"),
    skip_existing=False,
    show_in_notebook=SHOW_IN_NOTEBOOK,
    include_efficiency=INCLUDE_EFFICIENCY,
    save_merged_payload=SAVE_MERGED_PAYLOAD,
    save_final_panel=SAVE_FINAL_PANEL,
    cosmic_estimate=COSMIC_ESTIMATE,
    figsize=(28, 40),
    ncols=5,
)

# Non-interactive fallback (same as scripts/run_event_selection_batched.py)
batch_cfg = EventSelectionBatchedConfig(
    work_base=batch_work_base or default_event_selection_batched_work_root(today_str),
    plots_dir=batch_plots_dir,
    max_job_bytes=int(1.0 * 1024**3),
    max_files_per_sample=MAX_FILES_PER_SAMPLE,
    base_dir=base_dir,
    sample_dirs=BATCH_SAMPLE_DIRS,
    filename_str=FILENAME_STR,
    mc_univ_syst=("Flux", "G4", "GENIE"),
    skip_existing_batches=True,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    cosmic_estimate=COSMIC_ESTIMATE,
)

print(
    f"Live batched config: max_files_per_sample={MAX_FILES_PER_SAMPLE}  "
    f"update_every_n_files={UPDATE_EVERY_N_FILES}  "
    f"base_dir={base_dir}  work={live_cfg.work_base}"
)
for s, d in BATCH_SAMPLE_DIRS.items():
    print(f"  {s}: {d}")
assert live_cfg.base_dir and live_cfg.sample_dirs, (
    "live_cfg missing base_dir/sample_dirs — re-run the knobs cell"
)


In [ ]:
batch_result = None
live_result = None
merged_payload = None

# Interactive path: accumulate stats file-by-file and refresh the big panel.
# Skip this cell if you already have merged_histdata.pkl — the follow-up
# section can reload it via MERGED_HISTDATA_PKL / plots_dir.
live_result = run_live_accumulate(live_cfg)

save_fig_dir = str(live_result.plots_dir)
save_fig = True
show_plot = True
pot_str = live_result.pot_str
data_tot_pot = live_result.data_pot
merged_payload = live_result.merged_payload

print("Live batched workflow complete.")
print("  batches :", live_result.batches_dir)
print("  plots   :", live_result.plots_dir)
print("  payload :", live_result.payload_path)
print("  panel   :", live_result.panel_path)
print("  files   :", live_result.n_files_done)
print("  POT     :", pot_str)
if live_result.failed:
    print(f"  FAILED  : {len(live_result.failed)} job(s)")


## Summary, overlays & efficiency

Load `merged_histdata.pkl` from the live run (or set `MERGED_HISTDATA_PKL`) and
re-render plots. Toggle cosmics with `COSMIC_ESTIMATE = "intime"` or `"offbeam"`.


In [ ]:
# --- follow-up knobs (batched) ---
# Reload without re-running the live cell: set a path, or leave None to use
# in-memory merged_payload / {plots_dir}/merged_histdata.pkl
MERGED_HISTDATA_PKL = None  # e.g. path.join(plots_dir, "merged_histdata.pkl")

# Override cosmic estimate here if you want offbeam without re-editing the knobs cell
# COSMIC_ESTIMATE = "offbeam"

STAGE_LABELS = {
    "allreco": "All reconstructed slices",
    "is_clear_cosmic": "Not clear cosmic",
    "vertex_in_fv": "Vertex in Gen-1 fiducial volume",
    "nu_score": "Nu-score > {}".format(NU_SCORE_TH),
    "2prong": "Has exactly 2 PFPs",
    "2prong-contained": "Both PFPs per-TPC contained",
    "2prong-trackscore": "Both PFPs have track score > {}".format(TRACKSCORE_TH),
    "2prong-vtxdist": "Both track \n(start position - vertex) < {} cm".format(VTXDIST_TH),
    "2prong-muX": "One track is muon-like",
    "2prong-mup": "The other is proton-like",
}

from analysis_village.numucc_1p0pi.event_selection_live import (
    resolve_merged_payload,
    render_followup_from_payload,
    replot_from_payload,
)

_mp = globals().get("merged_payload")
_lr = globals().get("live_result")
merged_payload, payload_path = resolve_merged_payload(
    merged_payload=_mp,
    live_result=_lr,
    plots_dir=globals().get("save_fig_dir") or plots_dir,
    merged_histdata_pkl=MERGED_HISTDATA_PKL,
)
save_fig_dir = str(
    getattr(_lr, "plots_dir", None)
    or (payload_path.parent if payload_path.suffix == ".pkl" else None)
    or plots_dir
)
pot_str = merged_payload.get("pot_str") or globals().get("pot_str", "dummy")
save_fig = True
show_plot = True
approval = globals().get("approval", "")
print(f"Follow-up from: {payload_path}")
print(f"  save_fig_dir     = {save_fig_dir}")
print(f"  pot_str          = {pot_str}")
print(f"  COSMIC_ESTIMATE  = {COSMIC_ESTIMATE}")
print(f"  n_files_done     = {merged_payload.get('n_files_done')}")
print(f"  histdata plots   = {len(merged_payload['merged'].get('histdata', {}))}")
print(f"  bar stages       = {sorted(merged_payload['merged'].get('bar', {}))}")
else:
    df_dict = samples.df_dict  # MC stage history for efficiency
    print(list(df_dict.keys()))
    stage_labels = [STAGE_LABELS[k] for k in df_dict.keys()]


In [ ]:
followup = render_followup_from_payload(
    merged_payload,
    save_fig_dir=save_fig_dir,
    cosmic_estimate=COSMIC_ESTIMATE,
    save_fig=save_fig,
    show_fig=show_plot,
    render_overlays=True,
    render_summary=True,
    render_efficiency=True,
)
print("Wrote follow-up plots under", followup["save_fig_dir"])


In [ ]:
# Optional: rebuild the multi-panel live figure from the payload (no reprocessing)
fig = replot_from_payload(merged_payload, cosmic_estimate=COSMIC_ESTIMATE)
plt.show()
else:
    save_fig_dir


## Efficiency curves

Follow-up above already wrote `efficiency-*.png` and `eff_dict.pkl`.
Re-run the final-efficiency section to plot **only** the final (`2prong-mup`) curve
per variable from the saved payload.


# Final efficiency plots (from saved results)

Reload ``merged_histdata.pkl`` from the batched / live run above (or set
``FINAL_EFF_MERGED_PKL`` / ``MERGED_HISTDATA_PKL``) and write **one plot per
variable** of the **final** (``2prong-mup`` / sel_mup) selection efficiency only.
Black markers/lines, no legend. Skips reprocessing.


In [ ]:
# Reload saved stats and plot only the final (2prong-mup / sel_mup) efficiency.
FINAL_EFF_MERGED_PKL = None  # e.g. path.join(plots_dir, "merged_histdata.pkl")
FINAL_EFF_SAVE_DIR = None    # None → same dir as the payload / save_fig_dir
FINAL_EFF_STAGE = "2prong-mup"  # sel_mup
FINAL_EFF_SHOW = True
FINAL_EFF_COLOR = "black"

from analysis_village.numucc_1p0pi.event_selection_live import (
    resolve_merged_payload,
    ordered_efficiency_vars,
)
from analysis_village.numucc_1p0pi.utils import (
    add_approval_text,
    dpi,
    fig_ext,
    format_singlebin_plot,
    get_eff_err,
)

_mp = globals().get("merged_payload")
_lr = globals().get("live_result")
_eff_payload, _eff_payload_path = resolve_merged_payload(
    merged_payload=_mp,
    live_result=_lr,
    plots_dir=globals().get("save_fig_dir") or plots_dir,
    merged_histdata_pkl=FINAL_EFF_MERGED_PKL or globals().get("MERGED_HISTDATA_PKL"),
)

_eff_save_dir = FINAL_EFF_SAVE_DIR or str(
    getattr(_lr, "plots_dir", None)
    or globals().get("save_fig_dir")
    or (_eff_payload_path.parent if _eff_payload_path.suffix == ".pkl" else plots_dir)
)
makedirs(_eff_save_dir, exist_ok=True)

merged = _eff_payload["merged"]
eff = merged.get("eff") or {}
stage_keys = list(merged.get("stage_keys") or [])
approval = globals().get("approval", "internal")

if FINAL_EFF_STAGE not in eff:
    raise KeyError(
        f"Final stage {FINAL_EFF_STAGE!r} missing from payload eff stages: {sorted(eff)}"
    )

var_configs = ordered_efficiency_vars(merged)
print(f"Loading efficiencies from: {_eff_payload_path}")
print(f"  final stage = {FINAL_EFF_STAGE}")
print(f"  n variables = {len(var_configs)}")
print(f"  save dir    = {_eff_save_dir}")

final_eff_dict = {}
for var_config in var_configs:
    var_save_name = var_config.var_save_name

    # Denominator: generated signal on first efficiency stage (usually allreco).
    denom_stage = next((sk for sk in stage_keys if sk in eff and var_save_name in eff[sk]), None)
    if denom_stage is None or var_save_name not in eff[FINAL_EFF_STAGE]:
        print(f"  skip {var_save_name}: missing denom or final-stage counts")
        continue

    denom = eff[denom_stage][var_save_name]
    n_tot_pot = np.asarray(getattr(denom, "n_truth_nu_pot", denom.n_signal_pot), dtype=float)
    n_tot_raw = np.asarray(getattr(denom, "n_truth_nu_raw", denom.n_signal_raw), dtype=float)
    if float(np.sum(n_tot_raw)) <= 0.0 and float(np.sum(n_tot_pot)) <= 0.0:
        n_tot_pot = np.asarray(denom.n_signal_pot, dtype=float)
        n_tot_raw = np.asarray(denom.n_signal_raw, dtype=float)

    ea = eff[FINAL_EFF_STAGE][var_save_name]
    n_pot = np.asarray(ea.n_signal_pot, dtype=float)
    n_succ = np.asarray(ea.n_signal_raw, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        this_eff = np.where(n_tot_pot > 0, n_pot / n_tot_pot, 0.0)

    # Binomial (Wilson) uncertainty from raw pass/total counts — same as get_eff_err / plot_efficiency.
    ok = n_tot_raw > 0
    this_eff_err = [np.zeros(len(this_eff)), np.zeros(len(this_eff))]
    if np.any(ok):
        err_ok = get_eff_err(n_succ[ok], n_tot_raw[ok])
        this_eff_err[0][ok] = err_ok[0]
        this_eff_err[1][ok] = err_ok[1]

    bins = np.asarray(var_config.bins, dtype=float)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    denom_int_pot = float(np.sum(n_tot_pot))
    eff_int_pct = (float(ea.n_total_signal_int) / denom_int_pot * 100.0) if denom_int_pot > 0 else 0.0

    fig, ax = plt.subplots()
    ax.errorbar(
        bin_centers,
        this_eff,
        yerr=this_eff_err,
        fmt="o",
        linestyle="none",
        color=FINAL_EFF_COLOR,
        ecolor=FINAL_EFF_COLOR,
        markersize=5,
        capsize=2,
    )
    ax.set_xlabel(var_config.var_labels[0])
    ax.set_ylabel("Efficiency")
    ax.set_xlim(bins[0], bins[-1])
    y_top = float(np.nanmax(this_eff + np.asarray(this_eff_err[1], dtype=float)))
    ax.set_ylim(0, y_top * 1.1 if y_top > 0 else 0.1)

    add_approval_text(approval, 0.98, 0.97, "right", fontsize=14)
    if var_save_name == "integrated":
        format_singlebin_plot()

    save_name = path.join(_eff_save_dir, f"efficiency-final-{var_save_name}")
    plt.savefig(save_name + fig_ext, bbox_inches="tight", dpi=dpi)
    if FINAL_EFF_SHOW:
        plt.show()
    else:
        plt.close()

    final_eff_dict[var_save_name] = {
        "eff": this_eff,
        "eff_err": this_eff_err,
        "eff_int_pct": eff_int_pct,
        "stage": FINAL_EFF_STAGE,
    }
    print(f"  wrote {save_name}{fig_ext}  (integ. eff = {eff_int_pct:.2f}%)")

with open(path.join(_eff_save_dir, "eff_dict_final.pkl"), "wb") as f:
    pickle.dump(final_eff_dict, f)
print(f"Wrote {len(final_eff_dict)} final-efficiency plots under {_eff_save_dir}")
